# Base 3 — Limpeza e preparação dos dados

**Base:** qualidade do ar em Pequim, estação Aotizhongxin.  
**Alvo:** `PM2.5`.

Este notebook faz a ingestão, auditoria de qualidade, tratamento estrutural e preparação da série para a etapa de modelagem. A limpeza não faz imputação automática do alvo `PM2.5`, porque preencher o alvo com informação futura poderia gerar leakage.


In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

DATA_PATH = Path("../../bases/grupo3/PRSA_Data_Aotizhongxin_20130301-20170228.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {DATA_PATH.resolve()}\n"
        "Coloque este notebook em trabalho/notebooks/exploracao/"
    )

df_raw = pd.read_csv(DATA_PATH)
print("Arquivo:", DATA_PATH)
print("SHA-256:", hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())
print("Dimensão:", df_raw.shape)
display(df_raw.head())


Arquivo: ..\..\bases\grupo3\PRSA_Data_Aotizhongxin_20130301-20170228.csv
SHA-256: 38155611e0e2dc738cbc546855d8170fb0e39c31243406b07a175dd7b9878b31
Dimensão: (35064, 18)


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


In [2]:
# 1. Conversão de tipos e criação da chave temporal
df = df_raw.copy()

date_cols = ["year", "month", "day", "hour"]
for col in date_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

numeric_cols = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
                "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["datetime"] = pd.to_datetime(
    df[["year", "month", "day", "hour"]],
    errors="coerce"
)

df["wd"] = df["wd"].astype("string")
df["station"] = df["station"].astype("string")

df = df.sort_values("datetime", kind="stable").reset_index(drop=True)

quality = pd.DataFrame({
    "métrica": [
        "linhas", "colunas", "datas nulas", "datas duplicadas",
        "linhas exatamente duplicadas", "primeira data", "última data"
    ],
    "valor": [
        len(df), df.shape[1], df["datetime"].isna().sum(),
        df["datetime"].duplicated().sum(), df.duplicated().sum(),
        df["datetime"].min(), df["datetime"].max()
    ]
})
display(quality)


,métrica,valor
0,linhas,35064
1,colunas,19
2,datas nulas,0
3,datas duplicadas,0
4,linhas exatamente duplicadas,0
5,primeira data,2013-03-01 00:00:00
6,última data,2017-02-28 23:00:00


In [3]:
# 2. Auditoria de valores ausentes
missing = (
    df.isna().sum()
      .rename("nulos")
      .to_frame()
)
missing["percentual"] = 100 * missing["nulos"] / len(df)
display(missing.sort_values("nulos", ascending=False))

# A grade temporal está completa no arquivo original.
expected = pd.date_range(df["datetime"].min(), df["datetime"].max(), freq="h")
observed = pd.DatetimeIndex(df["datetime"].dropna().unique()).sort_values()
missing_timestamps = expected.difference(observed)

print("Horas esperadas:", len(expected))
print("Horas observadas:", len(observed))
print("Horas faltantes na grade:", len(missing_timestamps))


,nulos,percentual
CO,1776,5.065024
O3,1719,4.902464
NO2,1023,2.917522
SO2,935,2.666553
PM2.5,925,2.638033
PM10,718,2.047684
wd,81,0.231006
DEWP,20,0.057039
RAIN,20,0.057039
TEMP,20,0.057039


Horas esperadas: 35064
Horas observadas: 35064
Horas faltantes na grade: 0


In [4]:
# 3. Tratamento estrutural
# Não há linhas duplicadas exatas nem timestamps duplicados na Base 3.
assert df["datetime"].notna().all()
assert not df["datetime"].duplicated().any()

# station é constante e não é feature útil neste arquivo.
station_values = df["station"].dropna().unique()
print("Estações:", station_values)

# No alvo, não removemos outliers automaticamente:
# picos de PM2.5 são parte do fenômeno que queremos estudar.
q1, q3 = df["PM2.5"].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
outliers_pm25 = ((df["PM2.5"] > upper)).sum()

print(f"Limite superior de 1,5 IQR para PM2.5: {upper:.2f}")
print(f"Observações acima desse limite: {outliers_pm25:,}")


Estações: <StringArray>
['Aotizhongxin']
Length: 1, dtype: string
Limite superior de 1,5 IQR para PM2.5: 252.00
Observações acima desse limite: 1,624


In [5]:
# 4. Tabela limpa e série-alvo
# Mantemos NaN no alvo e nas covariáveis para que a estratégia de imputação
# seja definida posteriormente, dentro do protocolo temporal.
clean_cols = [
    "datetime", "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "wd", "WSPM"
]
df_clean = df[clean_cols].copy()

# Variáveis de calendário conhecidas antecipadamente
df_clean["hour_of_day"] = df_clean["datetime"].dt.hour
df_clean["day_of_week"] = df_clean["datetime"].dt.dayofweek
df_clean["month"] = df_clean["datetime"].dt.month

df_clean["hour_sin"] = np.sin(2 * np.pi * df_clean["hour_of_day"] / 24)
df_clean["hour_cos"] = np.cos(2 * np.pi * df_clean["hour_of_day"] / 24)
df_clean["month_sin"] = np.sin(2 * np.pi * df_clean["month"] / 12)
df_clean["month_cos"] = np.cos(2 * np.pi * df_clean["month"] / 12)

# Direção do vento: conversão circular, sem tratar N, NE, ... como escala ordinal.
wind_deg = df_clean["wd"].map({
    "N": 0, "NNE": 22.5, "NE": 45, "ENE": 67.5,
    "E": 90, "ESE": 112.5, "SE": 135, "SSE": 157.5,
    "S": 180, "SSW": 202.5, "SW": 225, "WSW": 247.5,
    "W": 270, "WNW": 292.5, "NW": 315, "NNW": 337.5
})
df_clean["wd_sin"] = np.sin(np.deg2rad(wind_deg))
df_clean["wd_cos"] = np.cos(np.deg2rad(wind_deg))

print("Linhas da tabela preparada:", len(df_clean))
display(df_clean.head())


Linhas da tabela preparada: 35064


,datetime,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,...,WSPM,hour_of_day,day_of_week,month,hour_sin,hour_cos,month_sin,month_cos,wd_sin,wd_cos
0,2013-03-01 00:00:00,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,...,4.4,0,4,3,0.000000,1.000000,1.0,6.123234e-17,-0.382683,0.923880
1,2013-03-01 01:00:00,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,...,4.7,1,4,3,0.258819,0.965926,1.0,6.123234e-17,0.000000,1.000000
2,2013-03-01 02:00:00,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,...,5.6,2,4,3,0.500000,0.866025,1.0,6.123234e-17,-0.382683,0.923880
3,2013-03-01 03:00:00,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,...,3.1,3,4,3,0.707107,0.707107,1.0,6.123234e-17,-0.707107,0.707107
4,2013-03-01 04:00:00,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,...,2.0,4,4,3,0.866025,0.500000,1.0,6.123234e-17,0.000000,1.000000


In [6]:
# 5. Separação cronológica para futura modelagem
# O corte é somente estrutural; o walk-forward será feito depois.
target_df = df_clean.dropna(subset=["PM2.5"]).copy()

split = int(len(target_df) * 0.80)
train_df = target_df.iloc[:split].copy()
test_df = target_df.iloc[split:].copy()

split_info = pd.DataFrame({
    "conjunto": ["treino", "teste"],
    "linhas": [len(train_df), len(test_df)],
    "proporção": [len(train_df)/len(target_df), len(test_df)/len(target_df)],
    "início": [train_df["datetime"].min(), test_df["datetime"].min()],
    "fim": [train_df["datetime"].max(), test_df["datetime"].max()]
})
display(split_info)

assert train_df["datetime"].max() < test_df["datetime"].min()

# Saídas reutilizáveis
df_clean.to_csv("base3_limpa_preparada.csv", index=False)
train_df.to_csv("base3_treino_preparada.csv", index=False)
test_df.to_csv("base3_teste_preparada.csv", index=False)

print("Arquivos CSV salvos na pasta de execução do notebook.")


,conjunto,linhas,proporção,início,fim
0,treino,27311,0.799994,2013-03-01 00:00:00,2016-05-16 01:00:00
1,teste,6828,0.200006,2016-05-16 02:00:00,2017-02-28 23:00:00


Arquivos CSV salvos na pasta de execução do notebook.


## Decisões de preparação

- `datetime` foi criado a partir de `year`, `month`, `day` e `hour`.
- A série possui grade horária completa; não foi necessário reindexar para criar timestamps.
- Não existem linhas exatamente duplicadas nem timestamps duplicados.
- `station` é constante e foi retirada da tabela de modelagem.
- `PM2.5` não teve outliers removidos automaticamente.
- Valores ausentes foram preservados para que a estratégia de imputação seja definida de forma causal na modelagem.
- Variáveis contemporâneas não devem ser usadas diretamente para prever o próprio `PM2.5` em `t`; se entrarem em modelos multivariados, devem respeitar a disponibilidade temporal.
- Para Holt-Winters, o modelo deve permanecer univariado e qualquer tratamento de valores ausentes do alvo precisa ser explicitamente justificado.
